# Skill 45: End-to-End Nano-GPT

**Phase 3 - LLMs & Generative AI Applications (Skills 31-45)**

## What this skill is about
Assemble embeddings, causal self-attention and feed-forward layers into a decoder-only language model, train it and sample from it.

## Objectives
- Build a character-level tokenizer and training batches.
- Use causal attention so each position sees only previous/current tokens.
- Train next-character prediction with cross-entropy and generate autoregressively.

Run the cells top to bottom. Requires `torch`, `numpy` and `matplotlib` (`pip install torch numpy matplotlib`).

## Theory and core formulas
$$\mathcal L=-\frac1{BT}\sum_{b,t}\log p_\theta(x_{b,t+1}\mid x_{b,\le t}),\qquad \text{perplexity}=e^{\mathcal L}$$
Uniform guessing scores $\ln 19=2.944$ for this 19-character vocabulary; the model has 106,515 parameters.

## Implementation
Reference code from the lab manual (unchanged).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(11)
text = ("to be or not to be that is the question. "
        "whether tis nobler in the mind to suffer. ") * 80
chars = sorted(set(text))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
vocab_size = len(chars)
block_size, batch_size = 48, 32

def get_batch():
    starts = torch.randint(0, len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in starts])
    y = torch.stack([data[i+1:i+block_size+1] for i in starts])
    return x, y

class TinyGPT(nn.Module):
    def __init__(self, vocab_size, d_model=64, heads=4, max_len=64):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=heads, dim_feedforward=4*d_model,
            dropout=0.0, activation="gelu", batch_first=True, norm_first=True
        )
        self.blocks = nn.TransformerEncoder(layer, num_layers=2)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        positions = torch.arange(T, device=idx.device)
        x = self.token_emb(idx) + self.pos_emb(positions)[None, :, :]
        # True above diagonal means future positions are blocked.
        causal_mask = torch.triu(
            torch.ones(T, T, device=idx.device, dtype=torch.bool), diagonal=1
        )
        x = self.blocks(x, mask=causal_mask)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, vocab_size),
                                   targets.reshape(-1))
        return logits, loss

model = TinyGPT(vocab_size)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)
for step in range(300):
    xb, yb = get_batch()
    _, loss = model(xb, yb)
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    if step % 100 == 0:
        print(f"step={step} loss={loss.item():.3f}")

@torch.no_grad()
def generate(model, idx, max_new_tokens=200):
    model.eval()
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -block_size:]
        logits, _ = model(idx_cond)
        probs = torch.softmax(logits[:, -1, :], dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_id], dim=1)
    return idx

start = torch.tensor([[stoi["t"]]])
generated = generate(model, start, 200)[0].tolist()
print("".join(itos[i] for i in generated))


## What to check
**Expected result:** Trains without shape errors; generated text uses only vocabulary characters. The tiny repeated corpus is a smoke test, not real language quality.

## Pitfalls
No validation split, so low loss means memorisation. `generate` leaves the model in eval mode. `block_size` must not exceed `max_len`.